## Training + Hyperparameter tuning + Evaluation - LR

### Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    log_loss
)

random_state = 42

### Load datasets

In [2]:
both_df = pd.read_csv("both_intervals_features.csv")
p600_df = pd.read_csv("p600_features.csv")
n400_df = pd.read_csv("n400_features.csv")

### Splitting function

In [ ]:
def split(df, random_state):
    df = df.copy() # copy to not break anything

    df["label"] = (df["Cloze"] > 0).astype(int) # cloze = 0 -> class 0 else class = 1

    cloze = df["Cloze"]  # keep it for the cloze performance analysis

    X = df.drop(columns=["Cloze", "label", "trial_global", "Trial", "site", "subject_global"]) # only keep features for training
    y = df["label"] # target variable 
    groups = df["subject_global"] # for subject-based splitting and grid search

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state) # 80-20 split with subjects as group to avoid data leakage
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]
    groups_test = groups.iloc[test_idx]

    cloze_test = cloze.iloc[test_idx]  # also split original cloze to keep track of shuffling and splitting of each seed

    return X_train, X_test, y_train, y_test, groups_train, groups_test, cloze_test

### Grid for grid search

In [4]:
parameter_grid = [
    {
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
        "model__C": [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1, 2, 5],
        "model__max_iter": [5000]
    },
    {
        "model__penalty": ["l1"],
        "model__solver": ["liblinear", "saga"],
        "model__C": [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1, 2, 5],
        "model__max_iter": [5000]
    }
]

### Select most frequent parameters

In [ ]:
# identify the most frequently occurring CONFIGURATION
def select_best_params(best_params_list):
    params_tuple = [tuple(sorted(p.items())) for p in best_params_list]
    most_common = Counter(params_tuple).most_common(1)[0][0]
    return dict(most_common)

### Nested CV

In [ ]:
# Nested cross-validation and grid search on the training data with subject-based grouping

def nested_cv_lr(X_train, y_train, groups_train, parameter_grid, random_state):
    outer_cv = GroupKFold(n_splits=5) # outer split into folds for evaluation of chosen hyperparameters within the inner folds 

    outer_scores = [] # keep track of the scores and associated hyperparam.
    best_params_list = []

    pipeline = Pipeline([
        ("scaler", StandardScaler()), # Important for logistic regression
        ("model", LogisticRegression(random_state=random_state)) # for each seed
    ])

    # grid search loop for the inner folds
    for outer_train_idx, outer_val_idx in outer_cv.split(X_train, y_train, groups_train):

        # extract the training and validation folds for each iteration
        X_outer_train = X_train.iloc[outer_train_idx]
        X_outer_val   = X_train.iloc[outer_val_idx]

        y_outer_train = y_train.iloc[outer_train_idx]
        y_outer_val   = y_train.iloc[outer_val_idx]

        # also split the groups
        groups_outer_train = groups_train.iloc[outer_train_idx]

        inner_cv = GroupKFold(n_splits=5)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=parameter_grid,
            cv=inner_cv,
            scoring="balanced_accuracy", # select best hyperparameters based on balanced accuracy
            n_jobs=-1
        )

        # run grid search, keeping track of groups
        grid_search.fit(X_outer_train, y_outer_train, groups=groups_outer_train)
        
        # retrieve the best model found by inner CV
        best_model = grid_search.best_estimator_
        best_params_list.append(grid_search.best_params_)
        
        # evaluate on outer fold
        y_pred = best_model.predict(X_outer_val)
        score = balanced_accuracy_score(y_outer_val, y_pred)
        outer_scores.append(score)

    return (outer_scores, select_best_params(best_params_list))

### Final train and test function

In [ ]:
def evaluate_lr(X_train, X_test, y_train, y_test, best_params, random_state):
    final_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty=best_params["model__penalty"],
            solver=best_params["model__solver"],
            C=best_params["model__C"],
            max_iter=best_params["model__max_iter"],
            random_state=random_state
        ))
    ]) # extract the best hyperparameters

    final_lr.fit(X_train, y_train) # fit the model on the training data

    y_pred = final_lr.predict(X_test) # make predictions
    y_prob = final_lr.predict_proba(X_test)[:, 1] # probabilities assigned to the predictions
    test_score_lr = balanced_accuracy_score(y_test, y_pred) # calculate balanced accuracy

    # save all of the evaluation metrics results
    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "test_score_lr": test_score_lr,
        "classification_report": classification_report(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    return final_lr, results # rreturn the final model

### Both intervals (P6 + N4) - the entire pipeline

In [ ]:
import json
import joblib
from collections import Counter

results_list = [] # keep track of all the results and params
all_params = []

# Run the entire pipeline across 10 different seeds to investigate how stable the results are
for seed in range(10):
    print(f"\nRunning seed {seed}...")
    
    # split the data (take seed into account)
    X_train_both, X_test_both, y_train_both, y_test_both, groups_train_both, groups_test_both, cloze_test_both = split(both_df, seed)

    # nested CV for the selection of the hyperparameters
    outer_scores_both, best_params_both = nested_cv_lr(
        X_train_both, y_train_both, groups_train_both, parameter_grid, seed
    )

    # evaluate the final model with the best hyperparameters on the current seed
    final_lr_both, results_both = evaluate_lr(
        X_train_both, X_test_both, y_train_both, y_test_both, best_params_both, seed
    )

    # store the results per seed
    results_list.append({
        "seed": seed,
        "accuracy": results_both["accuracy"],
        "balanced_accuracy": results_both["balanced_accuracy"],
        "f1": results_both["f1"],
        "precision": results_both["precision"],
        "recall": results_both["recall"],
        "roc_auc": results_both["roc_auc"],
        "best_params": best_params_both
    })

    all_params.append(best_params_both) # also save the parameters associated with each seed

    temp_df = pd.DataFrame(results_list)
    temp_df.to_csv("lr_results_running.csv", index=False)

    # Save predictions for this seed for performance as a function of cloze analysis
    pred_df = pd.DataFrame({
        "y_true": y_test_both.values,
        "y_pred": results_both["y_pred"],
        "y_prob": results_both["y_prob"],
        "Cloze": cloze_test_both.values,
        "Seed": seed
    })

    pred_df.to_csv(f"lr_predictions_seed_{seed}.csv", index=False)

    # Save model for this seed
    joblib.dump(final_lr_both, f"lr_model_seed_{seed}.joblib")

    print(f"Seed {seed} done.")

# final save
results_df = pd.DataFrame(results_list)
results_df.to_csv("lr_results_final.csv", index=False)

# for printing summary of the results
summary = {
    "accuracy_mean": float(results_df["accuracy"].mean()),
    "accuracy_std": float(results_df["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df["roc_auc"].mean()),
    "roc_auc_std": float(results_df["roc_auc"].std())
}

with open("lr_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

# select the most frequently CONFIGURATION across 10 seeds
params_tuple = [tuple(sorted(p.items())) for p in all_params]
best_params_final = dict(Counter(params_tuple).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final)

with open("lr_best_params.json", "w") as f:
    json.dump(best_params_final, f, indent=4)

# print final results
print("\nFinal Results:")
print(results_df)

print("\nSummary:")
print(summary)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__C': 0.001, 'model__max_iter': 5000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.569013           0.568996  0.557175   0.546154  0.568650  0.605097   
1     1  0.610893           0.608758  0.580645   0.587794  0.573668  0.646350   
2     2  0.588267           0.588213  0.593783   0.596340  0.591249  0.630400   
3     3  0.589524           0.590069  0.587540   0.570386  0.605759  0.618473   
4     4  0.604455           0.604927  0.600608   0.619824  0.582547  0.641585   
5     5  0.578604           0.578303  0.594882   0.577320  

### P600 interval - the entire pipeline (comments only on the combined data above, but the code is exactly the same, just a differrent dataset)

In [ ]:
import json
import joblib
from collections import Counter

results_list_p6 = []
all_params_p6 = []

for seed in range(10):
    print(f"\nRunning seed {seed}...")

    X_train_p6, X_test_p6, y_train_p6, y_test_p6, groups_train_p6, groups_test_p6 = split(p600_df, seed)

    outer_scores_p6, best_params_p6 = nested_cv_lr(
        X_train_p6, y_train_p6, groups_train_p6, parameter_grid, seed
    )

    final_lr_p6, results_p6 = evaluate_lr(
        X_train_p6, X_test_p6, y_train_p6, y_test_p6, best_params_p6, seed
    )

    results_list_p6.append({
        "seed": seed,
        "accuracy": results_p6["accuracy"],
        "balanced_accuracy": results_p6["balanced_accuracy"],
        "f1": results_p6["f1"],
        "precision": results_p6["precision"],
        "recall": results_p6["recall"],
        "roc_auc": results_p6["roc_auc"],
        "best_params": best_params_p6
    })

    all_params_p6.append(best_params_p6)

    temp_df_p6 = pd.DataFrame(results_list_p6)
    temp_df_p6.to_csv("lr_results_running_p6.csv", index=False)

    pred_df_p6 = pd.DataFrame({
        "y_true": y_test_p6,
        "y_pred": results_p6["y_pred"],
        "y_prob": results_p6["y_prob"]
    })
    pred_df_p6.to_csv(f"lr_predictions_seed_{seed}_p6.csv", index=False)

    joblib.dump(final_lr_p6, f"lr_model_seed_{seed}_p6.joblib")

    print(f"Seed {seed} done.")

results_df_p6 = pd.DataFrame(results_list_p6)
results_df_p6.to_csv("lr_results_final_p6.csv", index=False)

summary_p6 = {
    "accuracy_mean": float(results_df_p6["accuracy"].mean()),
    "accuracy_std": float(results_df_p6["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df_p6["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df_p6["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df_p6["roc_auc"].mean()),
    "roc_auc_std": float(results_df_p6["roc_auc"].std())
}

with open("lr_summary_p6.json", "w") as f:
    json.dump(summary_p6, f, indent=4)

params_tuple_p6 = [tuple(sorted(p.items())) for p in all_params_p6]
best_params_final_p6 = dict(Counter(params_tuple_p6).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final_p6)

with open("lr_best_params_p6.json", "w") as f:
    json.dump(best_params_final_p6, f, indent=4)

print("\nFinal Results:")
print(results_df_p6)

print("\nSummary:")
print(summary_p6)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__C': 0.001, 'model__max_iter': 5000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.545554           0.546172  0.540033   0.521878  0.559497  0.570024   
1     1  0.541217           0.539722  0.513274   0.511411  0.515152  0.578555   
2     2  0.543183           0.543143  0.548578   0.551836  0.545358  0.585381   
3     3  0.566542           0.566000  0.550693   0.550998  0.550388  0.594493   
4     4  0.567730           0.568376  0.559509   0.583120  0.537736  0.611278   
5     5  0.554495           0.554373  0.562839   0.557073  

### N400 interval - the entire pipeline (comments only on the combined data above, but the code is exactly the same, just a differrent dataset)

In [ ]:
import json
import joblib
from collections import Counter

results_list_n4 = []
all_params_n4 = []

for seed in range(10):
    print(f"\nRunning seed {seed}...")

    X_train_n4, X_test_n4, y_train_n4, y_test_n4, groups_train_n4, groups_test_n4 = split(n400_df, seed)

    outer_scores_n4, best_params_n4 = nested_cv_lr(
        X_train_n4, y_train_n4, groups_train_n4, parameter_grid, seed
    )

    final_lr_n4, results_n4 = evaluate_lr(
        X_train_n4, X_test_n4, y_train_n4, y_test_n4, best_params_n4, seed
    )

    results_list_n4.append({
        "seed": seed,
        "accuracy": results_n4["accuracy"],
        "balanced_accuracy": results_n4["balanced_accuracy"],
        "f1": results_n4["f1"],
        "precision": results_n4["precision"],
        "recall": results_n4["recall"],
        "roc_auc": results_n4["roc_auc"],
        "best_params": best_params_n4
    })

    all_params_n4.append(best_params_n4)

    temp_df_n4 = pd.DataFrame(results_list_n4)
    temp_df_n4.to_csv("lr_results_running_n4.csv", index=False)

    pred_df_n4 = pd.DataFrame({
        "y_true": y_test_n4,
        "y_pred": results_n4["y_pred"],
        "y_prob": results_n4["y_prob"]
    })
    pred_df_n4.to_csv(f"lr_predictions_seed_{seed}_n4.csv", index=False)

    joblib.dump(final_lr_n4, f"lr_model_seed_{seed}_n4.joblib")

    print(f"Seed {seed} done.")

results_df_n4 = pd.DataFrame(results_list_n4)
results_df_n4.to_csv("lr_results_final_n4.csv", index=False)

summary_n4 = {
    "accuracy_mean": float(results_df_n4["accuracy"].mean()),
    "accuracy_std": float(results_df_n4["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df_n4["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df_n4["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df_n4["roc_auc"].mean()),
    "roc_auc_std": float(results_df_n4["roc_auc"].std())
}

with open("lr_summary_n4.json", "w") as f:
    json.dump(summary_n4, f, indent=4)

params_tuple_n4 = [tuple(sorted(p.items())) for p in all_params_n4]
best_params_final_n4 = dict(Counter(params_tuple_n4).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final_n4)

with open("lr_best_params_n4.json", "w") as f:
    json.dump(best_params_final_n4, f, indent=4)

print("\nFinal Results:")
print(results_df_n4)

print("\nSummary:")
print(summary_n4)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__C': 0.001, 'model__max_iter': 5000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.554828           0.552601  0.519435   0.535194  0.504577  0.575165   
1     1  0.578018           0.577349  0.557613   0.549139  0.566353  0.601879   
2     2  0.560565           0.561076  0.552297   0.573563  0.532551  0.590410   
3     3  0.547301           0.546512  0.527607   0.531461  0.523810  0.560583   
4     4  0.547863           0.549401  0.518281   0.568214  0.476415  0.575823   
5     5  0.548970           0.549161  0.540900   0.555672  